# CaptionLab training and artifact export

[Open this notebook in Google Colab](https://colab.research.google.com/github/SahilBh01r1769/image-captioning/blob/fix/product-inference-foundation/notebooks/CaptionLab_Colab.ipynb)

This notebook obtains the portable artifacts needed by CaptionLab: three trained `best.pt` checkpoints, the matching `vocabulary.pkl`, frozen split, run configurations, histories, and provenance. It can resume interrupted Drive runs and skips only runs whose required files are complete.

The final cell verifies every checkpoint through the raw-image inference loader and creates `CaptionLab_phase1_artifacts.zip` in Google Drive. It does not select a winning model or run test-set evaluation.

## 1. Start a GPU runtime

In Colab choose **Runtime → Change runtime type → T4 GPU**. The assertion below prevents an accidental CPU extraction run.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
print(torch.__version__, torch.cuda.get_device_name(0))

## 2. Mount Drive and fetch the reviewed experiment code

The commit is pinned so a later repository edit cannot silently change a resumed experiment. Checkpoints and metadata live in Drive; temporary compute files live in `/content`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, shutil, subprocess

REPO = Path('/content/CaptionLab')
DRIVE_ROOT = Path('/content/drive/MyDrive/CaptionLab')
EXPERIMENT_COMMIT = '7b2f2e2934c64022f7b5862ed701e39fffcbdb96'
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', 'https://github.com/SahilBh01r1769/image-captioning.git', str(REPO)], check=True)
subprocess.run(['git', 'checkout', EXPERIMENT_COMMIT], cwd=REPO, check=True)
subprocess.run(['pip', 'install', '-q', 'pytest', 'tqdm', 'kagglehub', 'Pillow>=10,<12', 'numpy>=1.26,<3'], check=True)

for name in ('models', 'splits', 'outputs'):
    target = DRIVE_ROOT / name
    target.mkdir(exist_ok=True)
    link = REPO / name
    if link.exists() or link.is_symlink():
        if link.is_dir() and not link.is_symlink(): shutil.rmtree(link)
        else: link.unlink()
    link.symlink_to(target, target_is_directory=True)
print('Pinned commit:', EXPERIMENT_COMMIT)

## 3. Download Flickr8k directly in Colab

No 1 GB upload from your computer is needed. If the shared feature cache does not exist yet, KaggleHub downloads and extracts the public Flickr8k dataset into temporary Colab storage. Only `captions.txt` is retained in Drive; after feature extraction, later sessions restore that small file and the feature cache without downloading the images again.

If you already placed the extracted dataset in `MyDrive/Flickr8k`, the cell uses it instead.

In [ ]:
import kagglehub

DRIVE_DATASET = Path('/content/drive/MyDrive/Flickr8k')  # optional existing copy
DRIVE_CACHE = DRIVE_ROOT / 'feature_cache' / 'flickr8k_resnet50_spatial.pt'
CAPTIONS_BACKUP = DRIVE_ROOT / 'dataset_metadata' / 'captions.txt'
LOCAL_DATASET = Path('/content/Flickr8k')

if (DRIVE_DATASET / 'Images').is_dir() and (DRIVE_DATASET / 'captions.txt').is_file():
    FLICKR8K_SOURCE = DRIVE_DATASET
elif DRIVE_CACHE.exists() and CAPTIONS_BACKUP.is_file():
    LOCAL_DATASET.mkdir(parents=True, exist_ok=True)
    shutil.copy2(CAPTIONS_BACKUP, LOCAL_DATASET / 'captions.txt')
    FLICKR8K_SOURCE = LOCAL_DATASET
else:
    download_root = Path(kagglehub.dataset_download('adityajn105/flickr8k'))
    candidates = [path.parent for path in download_root.rglob('captions.txt') if (path.parent / 'Images').is_dir()]
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one Flickr8k layout under {download_root}, found {len(candidates)}')
    FLICKR8K_SOURCE = candidates[0]
    CAPTIONS_BACKUP.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(FLICKR8K_SOURCE / 'captions.txt', CAPTIONS_BACKUP)

assert (FLICKR8K_SOURCE / 'captions.txt').is_file(), 'captions.txt was not found'
if not DRIVE_CACHE.exists():
    assert (FLICKR8K_SOURCE / 'Images').is_dir(), 'Images/ is required for first-time feature extraction'
(REPO / 'data').mkdir(exist_ok=True)
dataset_link = REPO / 'data' / 'Flickr8k'
if dataset_link.exists() or dataset_link.is_symlink():
    if dataset_link.is_dir() and not dataset_link.is_symlink(): shutil.rmtree(dataset_link)
    else: dataset_link.unlink()
dataset_link.symlink_to(FLICKR8K_SOURCE, target_is_directory=True)
print('Dataset:', FLICKR8K_SOURCE)

## 4. Extract the shared frozen features once

This is the only ResNet-heavy stage. The float16 cache is retained in Drive, then copied to local Colab storage for faster training. Expect a file around 1.6 GB for Flickr8k.

In [ ]:
LOCAL_CACHE = Path('/content/flickr8k_resnet50_spatial.pt')
DRIVE_CACHE.parent.mkdir(exist_ok=True)
if not DRIVE_CACHE.exists():
    subprocess.run(['python', 'extract_features.py', '--output', str(LOCAL_CACHE), '--batch_size', '64', '--workers', '2'], cwd=REPO, check=True)
    shutil.copy2(LOCAL_CACHE, DRIVE_CACHE)
elif not LOCAL_CACHE.exists():
    shutil.copy2(DRIVE_CACHE, LOCAL_CACHE)
print(f'Feature cache ready: {LOCAL_CACHE} ({LOCAL_CACHE.stat().st_size / 2**30:.2f} GiB)')

## 5. Run the smoke check

This uses two train and two validation batches. It checks the full data/model/checkpoint path; it is not a result. Confirm that caption loss is finite and that the run ends with an artifact path.

In [ ]:
SMOKE_RUNS = Path('/content/captionlab_smoke_runs')
subprocess.run([
    'python', 'train.py', '--experiment', 'experiments/attention.json',
    '--feature_cache', str(LOCAL_CACHE), '--runs_dir', str(SMOKE_RUNS),
    '--smoke', '--overwrite_smoke', '--rebuild_vocab'
], cwd=REPO, check=True)

## 6. Full runs, one cell at a time

A repeated cell resumes from `last.pt` if the runtime disconnected. A completed run is skipped. Start with the baseline. While it runs, watch whether validation loss stops improving while training loss continues falling—that is overfitting, not progress.

In [ ]:
import json
RUNS_DIR = DRIVE_ROOT / 'runs'
RUNS_DIR.mkdir(exist_ok=True)

def run_or_resume(config_name, run_name):
    status_path = RUNS_DIR / run_name / 'status.json'
    required = [
        RUNS_DIR / run_name / 'checkpoints' / 'best.pt',
        RUNS_DIR / run_name / 'config.json',
        RUNS_DIR / run_name / 'history.json',
        RUNS_DIR / run_name / 'provenance.json',
        status_path,
    ]
    if status_path.exists() and json.loads(status_path.read_text()).get('state') == 'completed':
        missing = [str(path) for path in required if not path.is_file()]
        if missing: raise RuntimeError(f'{run_name} says completed but is missing: {missing}')
        print(run_name, 'already completed and complete')
        return
    command = ['python', 'train.py', '--experiment', f'experiments/{config_name}.json', '--feature_cache', str(LOCAL_CACHE), '--runs_dir', str(RUNS_DIR)]
    if (RUNS_DIR / run_name / 'checkpoints' / 'last.pt').exists(): command.append('--resume')
    subprocess.run(command, cwd=REPO, check=True)

run_or_resume('baseline', 'baseline_seed42')

In [ ]:
run_or_resume('attention', 'attention_seed42')

Coverage uses a pre-registered coefficient of `0.1`. Compare the printed caption loss and coverage loss separately. A lower total loss is not evidence that coverage improved captions.

In [ ]:
run_or_resume('attention_coverage', 'attention_coverage_seed42')

## 7. Inspect training dynamics

Do not choose a model from test captions. This plot uses training and validation caption loss only. Record the best epoch and whether each run stopped early.

In [ ]:
import matplotlib.pyplot as plt
for run_name in ('baseline_seed42', 'attention_seed42', 'attention_coverage_seed42'):
    history_path = RUNS_DIR / run_name / 'history.json'
    if not history_path.exists(): continue
    history = json.loads(history_path.read_text())
    epochs = [row['epoch'] for row in history]
    plt.plot(epochs, [row['validation']['caption_loss'] for row in history], marker='o', label=run_name)
    best = min(history, key=lambda row: row['validation']['caption_loss'])
    print(run_name, 'best epoch', best['epoch'], 'val caption loss', round(best['validation']['caption_loss'], 4))
plt.xlabel('Epoch'); plt.ylabel('Validation caption loss'); plt.legend(); plt.grid(alpha=.2); plt.show()

## 8. Verify and package the Phase 1 artifacts

Run this after all three statuses say `completed`. It loads each real checkpoint using the corrected raw-image inference path, verifies the shared vocabulary, and creates a compact checksummed ZIP. The feature cache, Flickr8k images, `last.pt` files, and optimizer-only recovery artifacts are excluded.

If the three runs were already completed in an earlier Colab session, the notebook reuses them. Older portable checkpoints without the new explicit backbone field remain supported and are reported as legacy artifacts.

In [ ]:
import hashlib, zipfile
from inference import load_model, load_vocabulary

RUN_NAMES = ('baseline_seed42', 'attention_seed42', 'attention_coverage_seed42')
VOCAB_PATH = DRIVE_ROOT / 'models' / 'vocabulary.pkl'
vocab = load_vocabulary(str(VOCAB_PATH))
artifact_paths = [VOCAB_PATH]
artifact_paths.extend(sorted((DRIVE_ROOT / 'splits').glob('*.json')))

for run_name in RUN_NAMES:
    run_dir = RUNS_DIR / run_name
    status = json.loads((run_dir / 'status.json').read_text())
    if status.get('state') != 'completed': raise RuntimeError(f'{run_name} is not complete')
    checkpoint_path = run_dir / 'checkpoints' / 'best.pt'
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model, architecture = load_model(str(checkpoint_path), vocab, torch.device('cpu'))
    del model
    backbone = checkpoint.get('frozen_backbone', 'legacy IMAGENET1K_V1 contract')
    print(run_name, 'verified:', architecture, '| epoch', checkpoint['epoch'], '| backbone', backbone)
    artifact_paths.extend([
        checkpoint_path, run_dir / 'config.json', run_dir / 'history.json',
        run_dir / 'provenance.json', run_dir / 'status.json',
    ])

missing = [str(path) for path in artifact_paths if not path.is_file()]
if missing: raise RuntimeError(f'Required artifacts are missing: {missing}')
manifest = {
    'source_commit': EXPERIMENT_COMMIT,
    'files': {
        str(path.relative_to(DRIVE_ROOT)): {
            'bytes': path.stat().st_size,
            'sha256': hashlib.sha256(path.read_bytes()).hexdigest(),
        } for path in artifact_paths
    },
}
manifest_path = DRIVE_ROOT / 'phase1_artifact_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2) + '\n')

bundle = DRIVE_ROOT / 'CaptionLab_phase1_artifacts.zip'
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in artifact_paths + [manifest_path]:
        archive.write(path, path.relative_to(DRIVE_ROOT))
print('Bundle ready:', bundle, f'({bundle.stat().st_size / 2**20:.1f} MiB)')
print('Upload this ZIP for the final real-checkpoint validation.')